# SESSÃO 10 — PRÉ-PROCESSAMENTO DOS DADOS COM O PROTOCOLO V4

**Projeto:** Classificação supervisionada de eventos indesejados em sistemas de produção de petróleo — 3W Dataset 2.0.0.

**Bloco:** C1 — Aprendizado supervisionado.

## 1. Contextualização

Nas Sessões 01–09, foram realizados o reconhecimento do dataset, o particionamento inicial, análises exploratórias, experimentos de aprendizado não supervisionado, extração de atributos temporais e testes preliminares com Random Forest.

Entretanto, a Sessão 09 demonstrou que a exigência de três sensores completos simultaneamente limitava o aproveitamento dos dados, excluindo diversos poços do treinamento.

Posteriormente, foi elaborado o particionamento V4 ajustado, preservando os poços de teste originais e mantendo a independência entre os conjuntos.

## 2. Particionamento utilizado

- **Treinamento:** 535 arquivos de 28 poços.
- **Validação:** 176 arquivos de seis poços.
- **Teste:** 408 arquivos de seis poços.

O protocolo V4 foi verificado estruturalmente e será utilizado como referência no desenvolvimento do C1. Sua compatibilidade com o C2 está sendo avaliada separadamente.

O conjunto de teste permanecerá reservado para a avaliação final dos modelos.

## 3. Objetivo da Sessão 10

Investigar a disponibilidade dos sensores e definir uma estratégia de pré-processamento capaz de ampliar o aproveitamento dos dados temporais, mantendo a consistência metodológica e evitando vazamento de informações entre poços.

## 4. Sensores investigados

- P-TPT
- T-TPT
- P-ANULAR
- P-JUS-CKGL
- P-MON-CKP

## 5. Configuração inicial dos experimentos

Serão investigadas janelas temporais de 100 amostras, com deslocamento de 50 amostras, respeitando a regularidade temporal e a homogeneidade dos rótulos.

Inicialmente, serão comparadas duas condições:

- **Regra original:** três sensores obrigatoriamente completos.
- **Regra alternativa:** pelo menos um dos cinco sensores completo.

A regra alternativa ainda será avaliada antes de sua adoção definitiva.

## 6. Resultados esperados

Ao final desta sessão, pretende-se identificar a cobertura dos sensores, avaliar a recuperação de janelas, reconhecer limitações associadas aos dados ausentes e estabelecer critérios documentados para a construção das matrizes utilizadas nos modelos supervisionados.

In [ ]:

# SESSAO 10 - TAREFA 01
# Carregamento e conferencia do particionamento V4

import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Localizar a pasta oficial da V4 no Drive
raiz = Path("/content/drive/MyDrive")

pastas = [
    p for p in raiz.rglob("PROTOCOLO_V4_COMPARTILHADO")
    if p.is_dir()
]

print("Pastas compartilhadas encontradas:", len(pastas))

assert len(pastas) == 1, (
    "Verifique a localizacao da pasta: " + str(pastas)
)

pasta_v4 = pastas[0]
print("Pasta utilizada:", pasta_v4)

# Localizar os dois CSVs, inclusive com prefixo "Cópia de"
arquivos = list(
    pasta_v4.glob("*V4_Instancias_CANDIDATA_AJUSTADA.csv")
)
pocos = list(
    pasta_v4.glob("*V4_Pocos_CANDIDATA_AJUSTADA.csv")
)

assert len(arquivos) == 1, "Manifesto de instancias nao encontrado."
assert len(pocos) == 1, "Manifesto de pocos nao encontrado."

v4 = pd.read_csv(arquivos[0])
pocos_v4 = pd.read_csv(pocos[0])

# Conferir os campos obrigatorios
colunas = {"Arquivo", "Categoria", "Poco", "Conjunto"}
assert colunas.issubset(v4.columns)

# Conferir a independencia dos pocos
assert v4.groupby("Poco")["Conjunto"].nunique().eq(1).all()

# Conferir se os dois manifestos concordam
conjuntos = v4[["Poco", "Conjunto"]].drop_duplicates()

assert conjuntos.sort_values(["Poco", "Conjunto"]).reset_index(
    drop=True
).equals(
    pocos_v4[["Poco", "Conjunto"]]
    .drop_duplicates()
    .sort_values(["Poco", "Conjunto"])
    .reset_index(drop=True)
)

# Conferir quantidades da V4
resumo = v4.groupby("Conjunto").agg(
    Arquivos=("Arquivo", "size"),
    Pocos=("Poco", "nunique")
)

esperado = {
    "Treino": (535, 28),
    "Validacao": (176, 6),
    "Teste": (408, 6)
}

for conjunto, valores in esperado.items():
    assert tuple(resumo.loc[conjunto]) == valores

assert len(v4) == 1119
assert v4["Poco"].nunique() == 40

# Conferir a cobertura por categoria
cobertura = pd.crosstab(
    v4["Categoria"], v4["Conjunto"]
).reindex(
    index=range(10),
    columns=["Treino", "Validacao", "Teste"],
    fill_value=0
)

assert (cobertura["Treino"] > 0).all()
assert (cobertura["Validacao"] > 0).sum() == 7

print("\nRESUMO DO PARTICIONAMENTO V4:")
display(resumo)

print("\nDISTRIBUICAO DAS CATEGORIAS:")
display(cobertura)

print("\nTAREFA 01: VERIFICACOES CONCLUIDAS!")


Mounted at /content/drive
Pastas compartilhadas encontradas: 1
Pasta utilizada: /content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/PROTOCOLO_V4_COMPARTILHADO

RESUMO DO PARTICIONAMENTO V4:


,Arquivos,Pocos
Conjunto,,
Teste,408,6
Treino,535,28
Validacao,176,6



DISTRIBUICAO DAS CATEGORIAS:


Conjunto,Treino,Validacao,Teste
Categoria,,,
0,268,107,219
1,1,0,3
2,18,3,1
3,31,0,1
4,143,38,162
5,6,1,4
6,3,0,3
7,14,19,3
8,9,1,4



TAREFA 01: VERIFICACOES CONCLUIDAS!


In [ ]:

# SESSAO 10 - TAREFA 02
# Diagnostico inicial de disponibilidade dos sensores

import io
import requests
import numpy as np
import pandas as pd

sensores = [
    "P-TPT",
    "T-TPT",
    "P-ANULAR",
    "P-JUS-CKGL",
    "P-MON-CKP"
]

pocos_amostra = [
    "WELL-00002",
    "WELL-00008",
    "WELL-00023"
]

resultados = []

for poco in pocos_amostra:

    # Selecionar um arquivo do treinamento V4
    item = (
        v4[
            (v4["Poco"] == poco)
            & (v4["Conjunto"] == "Treino")
        ]
        .sort_values("Arquivo")
        .iloc[0]
    )

    # Acessar o arquivo original do dataset 3W
    url = (
        "https://raw.githubusercontent.com/petrobras/3W/"
        f"main/dataset/{item['Categoria']}/"
        f"{item['Arquivo']}"
    )

    resposta = requests.get(url, timeout=120)
    resposta.raise_for_status()

    dados = pd.read_parquet(
        io.BytesIO(resposta.content)
    )

    # Selecionar os cinco sensores investigados
    sinais = dados.reindex(columns=sensores).apply(
        pd.to_numeric, errors="coerce"
    )

    # Calcular a porcentagem de valores disponiveis
    disponibilidade = (
        np.isfinite(sinais.to_numpy(dtype=float))
        .mean(axis=0) * 100
    )

    linha = {
        "Poco": poco,
        "Categoria": int(item["Categoria"]),
        "Amostras": len(dados)
    }

    for sensor, percentual in zip(
        sensores, disponibilidade
    ):
        linha[sensor] = round(float(percentual), 2)

    resultados.append(linha)

tabela_sensores = pd.DataFrame(resultados)

print("DISPONIBILIDADE DOS SENSORES (%):")
display(tabela_sensores)

print("\nTAREFA 02 EXECUTADA")


DISPONIBILIDADE DOS SENSORES (%):


,Poco,Categoria,Amostras,P-TPT,T-TPT,P-ANULAR,P-JUS-CKGL,P-MON-CKP
0,WELL-00002,2,12721,100.0,100.0,100.0,0.0,0.0
1,WELL-00008,0,21209,0.0,0.0,0.0,100.0,0.0
2,WELL-00023,7,559989,0.0,0.0,0.0,100.0,100.0



TAREFA 02 EXECUTADA


In [ ]:

# SESSAO 10 - TAREFA 03
# Comparacao das regras de selecao de janelas

from numpy.lib.stride_tricks import sliding_window_view

tamanho = 100
passo = 50

resultados_janelas = []
distribuicao_rotulos = []

for poco in pocos_amostra:

    # Selecionar o mesmo arquivo da Tarefa 02
    item = (
        v4[
            (v4["Poco"] == poco)
            & (v4["Conjunto"] == "Treino")
        ]
        .sort_values("Arquivo")
        .iloc[0]
    )

    url = (
        "https://raw.githubusercontent.com/petrobras/3W/"
        f"main/dataset/{item['Categoria']}/"
        f"{item['Arquivo']}"
    )

    resposta = requests.get(url, timeout=120)
    resposta.raise_for_status()

    dados = pd.read_parquet(io.BytesIO(resposta.content))

    # Rotulos temporais originais
    rotulos = pd.to_numeric(
        dados["class"], errors="coerce"
    ).to_numpy(dtype=float)

    # Valores dos cinco sensores
    sinais = dados.reindex(columns=sensores).apply(
        pd.to_numeric, errors="coerce"
    ).to_numpy(dtype=float)

    # Pontos de inicio das janelas
    inicios = np.arange(
        0, len(dados) - tamanho + 1, passo
    )

    # Janelas com rotulo valido e homogeneo
    r = sliding_window_view(
        rotulos, tamanho
    )[inicios]

    rotulo_ok = (
        np.isfinite(r).all(axis=1)
        & (r == r[:, :1]).all(axis=1)
        & np.isin(
            r[:, 0],
            list(range(10)) + list(range(101, 110))
        )
    )

    # Verificar continuidade temporal (~1 Hz)
    delta = np.diff(dados.index.asi8) / 1e9

    t = sliding_window_view(
        delta, tamanho - 1
    )[inicios]

    tempo_ok = np.isclose(
        t, 1, atol=0.1
    ).all(axis=1)

    # Sensores completos em cada janela
    s = sliding_window_view(
        np.isfinite(sinais), tamanho, axis=0
    )[inicios]

    completos = s.all(axis=2)

    # Comparar as duas regras
    antigas = rotulo_ok & tempo_ok & completos[:, :3].all(axis=1)
    novas = rotulo_ok & tempo_ok & completos.any(axis=1)

    resultados_janelas.append({
        "Poco": poco,
        "Janelas_antigas": int(antigas.sum()),
        "Janelas_novas": int(novas.sum()),
        "Janelas_recuperadas": int(
            novas.sum() - antigas.sum()
        )
    })

    # Registrar os rotulos originais das janelas
    for rotulo, quantidade in zip(
        *np.unique(r[novas, 0].astype(int), return_counts=True)
    ):
        distribuicao_rotulos.append({
            "Poco": poco,
            "Rotulo_original": int(rotulo),
            "Janelas": int(quantidade)
        })

comparacao_janelas = pd.DataFrame(resultados_janelas)
tabela_rotulos = pd.DataFrame(distribuicao_rotulos)

print("COMPARACAO DAS REGRAS:")
display(comparacao_janelas)

print("\nDISTRIBUICAO DOS ROTULOS ORIGINAIS:")
display(tabela_rotulos)

print("\nTAREFA 03 EXECUTADA")



COMPARACAO DAS REGRAS:


,Poco,Janelas_antigas,Janelas_novas,Janelas_recuperadas
0,WELL-00002,177,177,0
1,WELL-00008,0,351,351
2,WELL-00023,0,11124,11124



DISTRIBUICAO DOS ROTULOS ORIGINAIS:


,Poco,Rotulo_original,Janelas
0,WELL-00002,0,74
1,WELL-00002,2,2
2,WELL-00002,102,101
3,WELL-00008,0,351
4,WELL-00023,0,627
5,WELL-00023,107,10497



TAREFA 03 EXECUTADA


In [ ]:

# SESSAO 10 - TAREFA 04
# Auditoria preliminar dos 28 pocos do treinamento V4

import io
import requests
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# Selecionar um arquivo de cada poco de treinamento
amostra = (
    v4[v4["Conjunto"] == "Treino"]
    .sort_values("Arquivo")
    .groupby("Poco", as_index=False)
    .head(1)
    .sort_values("Poco")
)

assert len(amostra) == 28

resultados_auditoria = []

for _, item in amostra.iterrows():

    poco = item["Poco"]
    print("Analisando:", poco)

    try:
        url = (
            "https://raw.githubusercontent.com/petrobras/3W/"
            f"main/dataset/{item['Categoria']}/"
            f"{item['Arquivo']}"
        )

        resposta = requests.get(url, timeout=180)
        resposta.raise_for_status()

        dados = pd.read_parquet(
            io.BytesIO(resposta.content)
        )

        sinais = dados.reindex(
            columns=sensores
        ).apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=float)

        rotulos = pd.to_numeric(
            dados["class"], errors="coerce"
        ).to_numpy(dtype=float)

        # Disponibilidade dos cinco sensores
        disponibilidade = (
            np.isfinite(sinais).mean(axis=0) * 100
        )

        # Inicializar contagens
        antigas = 0
        novas = 0

        if len(dados) >= tamanho:

            inicios = np.arange(
                0, len(dados) - tamanho + 1, passo
            )

            r = sliding_window_view(
                rotulos, tamanho
            )[inicios]

            rotulo_ok = (
                np.isfinite(r).all(axis=1)
                & (r == r[:, :1]).all(axis=1)
                & np.isin(
                    r[:, 0],
                    list(range(10)) + list(range(101, 110))
                )
            )

            delta = np.diff(dados.index.asi8) / 1e9

            t = sliding_window_view(
                delta, tamanho - 1
            )[inicios]

            tempo_ok = np.isclose(
                t, 1, atol=0.1
            ).all(axis=1)

            s = sliding_window_view(
                np.isfinite(sinais), tamanho, axis=0
            )[inicios]

            completos = s.all(axis=2)
            base = rotulo_ok & tempo_ok

            antigas = int(
                (base & completos[:, :3].all(axis=1)).sum()
            )

            novas = int(
                (base & completos.any(axis=1)).sum()
            )

        resultados_auditoria.append({
            "Poco": poco,
            "Categoria_arquivo": int(item["Categoria"]),
            "Sensores_acima_90%": int(
                (disponibilidade > 90).sum()
            ),
            "Janelas_antigas": antigas,
            "Janelas_novas": novas,
            "Status": "OK"
        })

    except Exception as erro:
        resultados_auditoria.append({
            "Poco": poco,
            "Categoria_arquivo": int(item["Categoria"]),
            "Sensores_acima_90%": np.nan,
            "Janelas_antigas": np.nan,
            "Janelas_novas": np.nan,
            "Status": f"ERRO: {str(erro)[:100]}"
        })

auditoria_v4 = pd.DataFrame(resultados_auditoria)

print("\nAUDITORIA PRELIMINAR POR POCO:")
display(auditoria_v4)

ok = auditoria_v4[auditoria_v4["Status"] == "OK"]

print("\nRESUMO:")
print("Arquivos analisados com sucesso:", len(ok))
print("Arquivos com erro:", len(auditoria_v4) - len(ok))
print("Pocos com janelas pela regra antiga:",
      int((ok["Janelas_antigas"] > 0).sum()))
print("Pocos com janelas pela regra nova:",
      int((ok["Janelas_novas"] > 0).sum()))

print("\nTAREFA 04 - AUDITORIA PRELIMINAR EXECUTADA")


Analisando: WELL-00002
Analisando: WELL-00007
Analisando: WELL-00008
Analisando: WELL-00009
Analisando: WELL-00011
Analisando: WELL-00012
Analisando: WELL-00013
Analisando: WELL-00014
Analisando: WELL-00020
Analisando: WELL-00021
Analisando: WELL-00022
Analisando: WELL-00023
Analisando: WELL-00025
Analisando: WELL-00026
Analisando: WELL-00027
Analisando: WELL-00028
Analisando: WELL-00029
Analisando: WELL-00031
Analisando: WELL-00032
Analisando: WELL-00034
Analisando: WELL-00035
Analisando: WELL-00036
Analisando: WELL-00037
Analisando: WELL-00038
Analisando: WELL-00039
Analisando: WELL-00040
Analisando: WELL-00041
Analisando: WELL-00042

AUDITORIA PRELIMINAR POR POCO:


,Poco,Categoria_arquivo,Sensores_acima_90%,Janelas_antigas,Janelas_novas,Status
0,WELL-00002,2,3,177,177,OK
1,WELL-00007,4,4,0,142,OK
2,WELL-00008,0,1,0,351,OK
3,WELL-00009,2,5,129,129,OK
4,WELL-00011,2,3,491,491,OK
5,WELL-00012,2,4,133,133,OK
6,WELL-00013,2,3,199,199,OK
7,WELL-00014,9,5,1343,1343,OK
8,WELL-00020,9,5,73,73,OK
9,WELL-00021,7,5,854,854,OK



RESUMO:
Arquivos analisados com sucesso: 28
Arquivos com erro: 0
Pocos com janelas pela regra antiga: 15
Pocos com janelas pela regra nova: 28

TAREFA 04 - AUDITORIA PRELIMINAR EXECUTADA


In [ ]:

# SESSAO 10 - TAREFA 05
# Auditoria completa da V4, em lotes de 40 arquivos

import io
import requests
import numpy as np
import pandas as pd
from pathlib import Path

# Encontrar a pasta do Notebook 10
pastas_exp02 = [
    p for p in Path("/content/drive/MyDrive").rglob(
        "EXP02_Preprocessamento_V4"
    ) if p.is_dir()
]

assert len(pastas_exp02) == 1, (
    f"Verifique as pastas encontradas: {pastas_exp02}"
)

saida = (
    pastas_exp02[0] /
    "10_Auditoria_535_Arquivos_V4.csv"
)

# Organizar os 535 arquivos de treinamento
treino = v4[v4["Conjunto"] == "Treino"].copy()
treino["Ordem"] = treino.groupby("Poco").cumcount()
treino = treino.sort_values(["Ordem", "Poco"])

assert len(treino) == 535

# Recuperar auditorias anteriores, caso existam
if saida.exists():
    auditoria = pd.read_csv(saida)
else:
    auditoria = pd.DataFrame()

ja_analisados = (
    set(auditoria["Arquivo"])
    if "Arquivo" in auditoria.columns else set()
)

lote = treino[
    ~treino["Arquivo"].isin(ja_analisados)
].head(40)

# Verificar se todos os elementos de uma janela sao validos
def janela_completa(valido, comprimento, inicios):
    valido = np.asarray(valido, dtype=bool)

    if valido.ndim == 1:
        valido = valido[:, None]

    erros = np.cumsum(
        (~valido).astype(np.int32),
        axis=0
    )

    erros = np.vstack([
        np.zeros((1, erros.shape[1]), dtype=np.int32),
        erros
    ])

    return (
        erros[inicios + comprimento] -
        erros[inicios]
    ) == 0

resultados = []

for _, item in lote.iterrows():

    arquivo = item["Arquivo"]
    print("Analisando:", arquivo)

    try:
        url = (
            "https://raw.githubusercontent.com/petrobras/3W/"
            f"main/dataset/{item['Categoria']}/{arquivo}"
        )

        resposta = requests.get(url, timeout=180)
        resposta.raise_for_status()
        dados = pd.read_parquet(io.BytesIO(resposta.content))

        sinais = dados.reindex(columns=sensores).apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=float)

        rotulos = pd.to_numeric(
            dados["class"], errors="coerce"
        ).to_numpy(dtype=float)

        inicios = np.arange(
            0, len(dados) - tamanho + 1, passo
        )

        antigas = novas = um = dois_mais = 0

        if len(inicios) > 0:

            # Rotulos permitidos e constantes na janela
            permitidos = np.isin(
                rotulos,
                list(range(10)) + list(range(101, 110))
            )

            rotulo_ok = (
                janela_completa(
                    permitidos, tamanho, inicios
                )[:, 0]
                &
                janela_completa(
                    rotulos[1:] == rotulos[:-1],
                    tamanho - 1, inicios
                )[:, 0]
            )

            # Regularidade temporal
            delta = np.diff(dados.index.asi8) / 1e9

            tempo_ok = janela_completa(
                np.isclose(delta, 1, atol=0.1),
                tamanho - 1, inicios
            )[:, 0]

            # Disponibilidade instrumental
            completos = janela_completa(
                np.isfinite(sinais),
                tamanho, inicios
            )

            base = rotulo_ok & tempo_ok
            quantidade = completos.sum(axis=1)

            antigas = int(
                (base & completos[:, :3].all(axis=1)).sum()
            )

            novas = int(
                (base & (quantidade >= 1)).sum()
            )

            um = int(
                (base & (quantidade == 1)).sum()
            )

            dois_mais = int(
                (base & (quantidade >= 2)).sum()
            )

        resultados.append({
            "Arquivo": arquivo,
            "Poco": item["Poco"],
            "Categoria": int(item["Categoria"]),
            "Amostras": len(dados),
            "Janelas_antigas": antigas,
            "Janelas_novas": novas,
            "Janelas_1_sensor": um,
            "Janelas_2_ou_mais": dois_mais,
            "Status": "OK"
        })

    except Exception as erro:
        resultados.append({
            "Arquivo": arquivo,
            "Poco": item["Poco"],
            "Categoria": int(item["Categoria"]),
            "Status": f"ERRO: {str(erro)[:100]}"
        })

# Salvar o lote sem perder resultados anteriores
auditoria = pd.concat(
    [auditoria, pd.DataFrame(resultados)],
    ignore_index=True
)

auditoria = auditoria.drop_duplicates(
    subset="Arquivo", keep="last"
)

auditoria.to_csv(saida, index=False)

ok = auditoria[auditoria["Status"] == "OK"]

print("\nPROGRESSO DA AUDITORIA:")
print("Arquivos registrados:", len(auditoria), "/ 535")
print("Arquivos processados com sucesso:", len(ok))
print("Arquivos com erro:",
      len(auditoria) - len(ok))

print("\nCOBERTURA ACUMULADA:")
print("Pocos com janelas pela regra original:",
      ok.loc[
          ok["Janelas_antigas"] > 0, "Poco"
      ].nunique())

print("Pocos com janelas pela regra alternativa:",
      ok.loc[
          ok["Janelas_novas"] > 0, "Poco"
      ].nunique())

print("\nRESUMO DO LOTE:")
display(pd.DataFrame(resultados).tail(10))

print("\nAUDITORIA PARCIAL SALVA EM:")
print(saida)


Analisando: WELL-00002_20170810160000.parquet
Analisando: WELL-00002_20170810210000.parquet
Analisando: WELL-00002_20170811020000.parquet
Analisando: WELL-00002_20170811070000.parquet
Analisando: WELL-00002_20170811120000.parquet
Analisando: WELL-00002_20170811170000.parquet
Analisando: WELL-00002_20170811220000.parquet
Analisando: WELL-00002_20170812030000.parquet
Analisando: WELL-00002_20170812080000.parquet
Analisando: WELL-00002_20170812130000.parquet
Analisando: WELL-00002_20170812180000.parquet
Analisando: WELL-00002_20170812230000.parquet
Analisando: WELL-00002_20170813040000.parquet
Analisando: WELL-00002_20170813090000.parquet
Analisando: WELL-00002_20170813140000.parquet

PROGRESSO DA AUDITORIA:
Arquivos registrados: 535 / 535
Arquivos processados com sucesso: 535
Arquivos com erro: 0

COBERTURA ACUMULADA:
Pocos com janelas pela regra original: 15
Pocos com janelas pela regra alternativa: 28

RESUMO DO LOTE:


,Arquivo,Poco,Categoria,Amostras,Janelas_antigas,Janelas_novas,Janelas_1_sensor,Janelas_2_ou_mais,Status
5,WELL-00002_20170811170000.parquet,WELL-00002,0,21547,357,357,0,357,OK
6,WELL-00002_20170811220000.parquet,WELL-00002,0,21506,357,357,0,357,OK
7,WELL-00002_20170812030000.parquet,WELL-00002,0,21560,358,358,0,358,OK
8,WELL-00002_20170812080000.parquet,WELL-00002,0,21520,357,357,0,357,OK
9,WELL-00002_20170812130000.parquet,WELL-00002,0,21142,349,349,0,349,OK
10,WELL-00002_20170812180000.parquet,WELL-00002,0,21439,355,355,0,355,OK
11,WELL-00002_20170812230000.parquet,WELL-00002,0,21506,357,357,0,357,OK
12,WELL-00002_20170813040000.parquet,WELL-00002,0,21560,358,358,0,358,OK
13,WELL-00002_20170813090000.parquet,WELL-00002,0,21520,357,357,0,357,OK
14,WELL-00002_20170813140000.parquet,WELL-00002,0,21528,357,357,0,357,OK



AUDITORIA PARCIAL SALVA EM:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/10_Auditoria_535_Arquivos_V4.csv


In [ ]:

# SESSAO 10 - TAREFA 06
# Consolidacao dos resultados da auditoria V4

import pandas as pd
from pathlib import Path

# Localizar o CSV da auditoria completa
caminhos = list(
    Path("/content/drive/MyDrive").rglob(
        "10_Auditoria_535_Arquivos_V4.csv"
    )
)

assert len(caminhos) == 1, (
    f"Verifique os arquivos encontrados: {caminhos}"
)

aud = pd.read_csv(caminhos[0])

# Conferir a integridade da auditoria
assert len(aud) == 535
assert aud["Arquivo"].is_unique
assert aud["Status"].eq("OK").all()

assert set(aud["Arquivo"]) == set(
    v4.loc[v4["Conjunto"] == "Treino", "Arquivo"]
)

assert (
    aud["Janelas_1_sensor"] +
    aud["Janelas_2_ou_mais"] ==
    aud["Janelas_novas"]
).all()

# 1. Comparacao geral das regras
resumo = pd.DataFrame({
    "Regra": ["Original", "Alternativa"],
    "Arquivos_com_janelas": [
        int((aud["Janelas_antigas"] > 0).sum()),
        int((aud["Janelas_novas"] > 0).sum())
    ],
    "Pocos_com_janelas": [
        aud.loc[
            aud["Janelas_antigas"] > 0, "Poco"
        ].nunique(),
        aud.loc[
            aud["Janelas_novas"] > 0, "Poco"
        ].nunique()
    ],
    "Total_janelas": [
        int(aud["Janelas_antigas"].sum()),
        int(aud["Janelas_novas"].sum())
    ]
})

print("1. COMPARACAO GERAL:")
display(resumo)

# 2. Quantidade de sensores por janela
total = aud["Janelas_novas"].sum()

composicao = pd.DataFrame({
    "Condicao": [
        "Somente 1 sensor",
        "2 ou mais sensores"
    ],
    "Janelas": [
        int(aud["Janelas_1_sensor"].sum()),
        int(aud["Janelas_2_ou_mais"].sum())
    ]
})

composicao["Percentual"] = (
    composicao["Janelas"] / total * 100
).round(2)

print("\n2. COMPOSICAO DAS JANELAS:")
display(composicao)

# 3. Distribuicao por poco
por_poco = aud.groupby("Poco").agg(
    Arquivos=("Arquivo", "size"),
    Janelas=("Janelas_novas", "sum")
).sort_values("Janelas", ascending=False)

por_poco["Percentual_total"] = (
    por_poco["Janelas"] / total * 100
).round(2)

print("\n3. DEZ POCOS COM MAIS JANELAS:")
display(por_poco.head(10))

# 4. Distribuicao por categoria do arquivo
por_categoria = aud.groupby("Categoria").agg(
    Arquivos=("Arquivo", "size"),
    Pocos=("Poco", "nunique"),
    Janelas=("Janelas_novas", "sum")
).sort_index()

por_categoria["Percentual_total"] = (
    por_categoria["Janelas"] / total * 100
).round(2)

print("\n4. DISTRIBUICAO POR CATEGORIA DO ARQUIVO:")
display(por_categoria)

# 5. Arquivos sem janelas validas
print("\n5. ARQUIVOS SEM JANELAS:")
print("Regra original:",
      int((aud["Janelas_antigas"] == 0).sum()))
print("Regra alternativa:",
      int((aud["Janelas_novas"] == 0).sum()))

print("\nTAREFA 06 EXECUTADA")


1. COMPARACAO GERAL:


,Regra,Arquivos_com_janelas,Pocos_com_janelas,Total_janelas
0,Original,434,15,220625
1,Alternativa,535,28,342004



2. COMPOSICAO DAS JANELAS:


,Condicao,Janelas,Percentual
0,Somente 1 sensor,20075,5.87
1,2 ou mais sensores,321929,94.13



3. DEZ POCOS COM MAIS JANELAS:


,Arquivos,Janelas,Percentual_total
Poco,,,
WELL-00002,326,89738,26.24
WELL-00022,8,61456,17.97
WELL-00023,4,30915,9.04
WELL-00026,3,25682,7.51
WELL-00008,57,20071,5.87
WELL-00041,9,19563,5.72
WELL-00014,55,16295,4.76
WELL-00025,1,14366,4.20
WELL-00027,1,13531,3.96



4. DISTRIBUICAO POR CATEGORIA DO ARQUIVO:


,Arquivos,Pocos,Janelas,Percentual_total
Categoria,,,,
0,268,3,92855,27.15
1,1,1,282,0.08
2,18,5,2939,0.86
3,31,1,10967,3.21
4,143,3,20213,5.91
5,6,1,7106,2.08
6,3,1,941,0.28
7,14,3,94118,27.52
8,9,7,72635,21.24



5. ARQUIVOS SEM JANELAS:
Regra original: 101
Regra alternativa: 0

TAREFA 06 EXECUTADA


In [ ]:

# SESSAO 10 - TAREFA 07
# Comparacao entre os criterios de disponibilidade

import pandas as pd

# Utilizar o DataFrame "aud" criado na Tarefa 06

criterios = {
    "3 sensores originais": "Janelas_antigas",
    "Minimo 2 sensores": "Janelas_2_ou_mais",
    "Minimo 1 sensor": "Janelas_novas"
}

resultados = []

for nome, coluna in criterios.items():

    validos = aud[aud[coluna] > 0]

    resultados.append({
        "Criterio": nome,
        "Arquivos": len(validos),
        "Pocos": validos["Poco"].nunique(),
        "Janelas": int(aud[coluna].sum())
    })

comparacao_criterios = pd.DataFrame(resultados)

print("1. COMPARACAO DOS CRITERIOS:")
display(comparacao_criterios)

# Avaliar quais pocos dependem de apenas um sensor
por_poco = aud.groupby("Poco").agg(
    Janelas_1_sensor=("Janelas_1_sensor", "sum"),
    Janelas_2_mais=("Janelas_2_ou_mais", "sum"),
    Total=("Janelas_novas", "sum")
)

por_poco["Percentual_1_sensor"] = (
    100 * por_poco["Janelas_1_sensor"] /
    por_poco["Total"]
).round(2)

print("\n2. POCOS COM JANELAS DE APENAS 1 SENSOR:")
display(
    por_poco[
        por_poco["Janelas_1_sensor"] > 0
    ].sort_values(
        "Janelas_1_sensor", ascending=False
    )
)

print("\n3. POCOS EXCLUIDOS COM MINIMO 2 SENSORES:")
excluidos = por_poco[
    por_poco["Janelas_2_mais"] == 0
]

display(excluidos)

# Avaliar a distribuicao por categoria de arquivo
por_categoria_sensores = aud.groupby("Categoria").agg(
    Janelas_1_sensor=("Janelas_1_sensor", "sum"),
    Janelas_2_mais=("Janelas_2_ou_mais", "sum")
)

print("\n4. DISTRIBUICAO POR CATEGORIA DO ARQUIVO:")
display(por_categoria_sensores)

print("\nTAREFA 07 EXECUTADA")


1. COMPARACAO DOS CRITERIOS:


,Criterio,Arquivos,Pocos,Janelas
0,3 sensores originais,434,15,220625
1,Minimo 2 sensores,478,27,321929
2,Minimo 1 sensor,535,28,342004



2. POCOS COM JANELAS DE APENAS 1 SENSOR:


,Janelas_1_sensor,Janelas_2_mais,Total,Percentual_1_sensor
Poco,,,,
WELL-00008,20071,0,20071,100.00
WELL-00014,2,16293,16295,0.01
WELL-00023,2,30913,30915,0.01



3. POCOS EXCLUIDOS COM MINIMO 2 SENSORES:


,Janelas_1_sensor,Janelas_2_mais,Total,Percentual_1_sensor
Poco,,,,
WELL-00008,20071,0,20071,100.0



4. DISTRIBUICAO POR CATEGORIA DO ARQUIVO:


,Janelas_1_sensor,Janelas_2_mais
Categoria,,
0,20071,72784
1,0,282
2,0,2939
3,0,10967
4,2,20211
5,0,7106
6,0,941
7,2,94116
8,0,72635



TAREFA 07 EXECUTADA


In [ ]:

# SESSAO 10 - TAREFA 08
# Impacto dos criterios sobre arquivos normais

# Selecionar arquivos de categoria 0
normais = aud[aud["Categoria"] == 0].copy()

# Comparar a contribuicao por poco
resumo_normais = normais.groupby("Poco").agg(
    Arquivos=("Arquivo", "size"),
    Arquivos_min_1=(
        "Janelas_novas",
        lambda x: int((x > 0).sum())
    ),
    Arquivos_min_2=(
        "Janelas_2_ou_mais",
        lambda x: int((x > 0).sum())
    ),
    Janelas_min_1=("Janelas_novas", "sum"),
    Janelas_min_2=("Janelas_2_ou_mais", "sum")
)

print("1. CONTRIBUICAO DOS POCOS COM ARQUIVOS NORMAIS:")
display(resumo_normais)

print("\n2. COBERTURA DOS ARQUIVOS NORMAIS:")

print(
    "Pocos com minimo 1 sensor:",
    normais.loc[
        normais["Janelas_novas"] > 0, "Poco"
    ].nunique()
)

print(
    "Pocos com minimo 2 sensores:",
    normais.loc[
        normais["Janelas_2_ou_mais"] > 0, "Poco"
    ].nunique()
)

print(
    "Janelas com minimo 1 sensor:",
    int(normais["Janelas_novas"].sum())
)

print(
    "Janelas com minimo 2 sensores:",
    int(normais["Janelas_2_ou_mais"].sum())
)

print("\nTAREFA 08 EXECUTADA")


1. CONTRIBUICAO DOS POCOS COM ARQUIVOS NORMAIS:


,Arquivos,Arquivos_min_1,Arquivos_min_2,Janelas_min_1,Janelas_min_2
Poco,,,,,
WELL-00002,209,209,209,72500,72500
WELL-00007,2,2,2,284,284
WELL-00008,57,57,0,20071,0



2. COBERTURA DOS ARQUIVOS NORMAIS:
Pocos com minimo 1 sensor: 3
Pocos com minimo 2 sensores: 2
Janelas com minimo 1 sensor: 92855
Janelas com minimo 2 sensores: 72784

TAREFA 08 EXECUTADA
